# Step 3 – Live Line Following

Loads the trained ResNet18 model and runs it on live camera frames to steer the robot.
Depth data from the ZED camera provides a **collision safety stop**.

**Prerequisites:** `line_follower.pth` must exist (produced by Step 2).

**System overview:**
```
ZED Camera → colour frame → ResNet18 → steering decision (left / forward / right)
           → depth frame  → safety check → override to STOP if obstacle < 300 mm
```

In [ ]:
# ── Cell 1: Load model ────────────────────────────────────────────────────────
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn

SAVE_PATH = 'line_follower.pth'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt        = torch.load(SAVE_PATH, map_location=device)
class_names = ckpt['class_names']   # ['forward', 'left', 'right'] — alphabetical
print('Classes:', class_names)
print(f'Checkpoint val_acc: {ckpt["val_acc"]:.3f}')

model = torchvision.models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()

# Same normalisation used during training
infer_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])
print('Model ready.')

In [ ]:
# ── Cell 2: Start camera ──────────────────────────────────────────────────────
import cv2, threading, time, numpy as np
import pyzed.sl as sl
import traitlets
from traitlets.config.configurable import SingletonConfigurable

class Camera(SingletonConfigurable):
    color_value = traitlets.Any()

    def __init__(self):
        super().__init__()
        self.zed = sl.Camera()
        init = sl.InitParameters()
        init.camera_resolution = sl.RESOLUTION.VGA
        init.depth_mode = sl.DEPTH_MODE.ULTRA
        init.coordinate_units = sl.UNIT.MILLIMETER
        status = self.zed.open(init)
        if status != sl.ERROR_CODE.SUCCESS:
            print('Camera open failed:', status)
            exit(1)
        self.runtime = sl.RuntimeParameters()
        self.thread_runnning_flag = False
        info = self.zed.get_camera_information()
        self.width  = info.camera_configuration.resolution.width
        self.height = info.camera_configuration.resolution.height
        self.image  = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)
        self.depth  = sl.Mat(self.width, self.height, sl.MAT_TYPE.F32_C1, sl.MEM.CPU)
        self.depth_image = np.zeros((self.height, self.width), dtype=np.float32)

    def _capture_frames(self):
        while self.thread_runnning_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                self.zed.retrieve_measure(self.depth, sl.MEASURE.DEPTH)
                raw = self.image.get_data()
                self.color_value = cv2.cvtColor(raw, cv2.COLOR_BGRA2BGR)
                self.depth_image = np.nan_to_num(
                    np.asanyarray(self.depth.get_data()),
                    nan=0.0).astype(np.float32)

    def start(self):
        if not self.thread_runnning_flag:
            self.thread_runnning_flag = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()

    def stop(self):
        if self.thread_runnning_flag:
            self.thread_runnning_flag = False
            self.thread.join()

def bgr8_to_jpeg(img):
    return bytes(cv2.imencode('.jpg', img)[1])

camera = Camera()
camera.start()
time.sleep(1)  # let camera warm up
print('Camera started.')

In [ ]:
# ── Cell 3: Live line-following loop ──────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display
import motors

robot = motors.MotorsYukon(mecanum=False)

# ── Tunable parameters ────────────────────────────────────────────────────────
FORWARD_SPEED   = 0.40    # straight-line speed (0–1)
TURN_SPEED      = 0.35    # turning speed
SAFETY_DIST_MM  = 300     # stop if anything closer than this in the central zone
CONFIDENCE_THR  = 0.60    # minimum softmax confidence to act (else stop)
RUN_FRAMES      = 500     # number of inference frames before auto-stop (safety)

# Display widgets
display_color = widgets.Image(format='jpeg', width='50%')
status_label  = widgets.Label(value='Starting...')
display(widgets.VBox([display_color, status_label]))

# ── Helper: depth safety check ────────────────────────────────────────────────
def obstacle_too_close(depth_img):
    """Return True if minimum depth in the central frontal zone < SAFETY_DIST_MM."""
    h, w = depth_img.shape
    zone = depth_img[h//4 : 3*h//4, w//4 : 3*w//4].copy()
    zone[zone < 50]   = 0  # ignore noise / invalid readings
    zone[zone > 3000] = 0  # ignore far-away objects
    nonzero = zone[zone != 0]
    if nonzero.size == 0:
        return False
    return nonzero.min() < SAFETY_DIST_MM

# ── Helper: run one CNN inference step ────────────────────────────────────────
def predict_steering(frame):
    """Return (class_name, confidence) for the current frame."""
    h = frame.shape[0]
    # Use the bottom half of the frame (same crop as training)
    crop = frame[h//2:, :]
    rgb  = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    x    = infer_tf(rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
    probs  = torch.softmax(logits, dim=1)[0]
    idx    = probs.argmax().item()
    return class_names[idx], probs[idx].item()

# ── Main inference loop ───────────────────────────────────────────────────────
print('Running line follower. Interrupt kernel to stop.')
try:
    for frame_idx in range(RUN_FRAMES):
        if camera.color_value is None:
            time.sleep(0.05)
            continue

        frame = camera.color_value.copy()

        # 1. Safety check using depth
        if obstacle_too_close(camera.depth_image):
            robot.stop()
            action = 'STOP (obstacle)'
            conf   = 1.0
        else:
            # 2. CNN steering decision
            steering, conf = predict_steering(frame)

            if conf < CONFIDENCE_THR:
                robot.stop()
                action = f'STOP (low conf {conf:.2f})'
            elif steering == 'forward':
                robot.forward(FORWARD_SPEED)
                action = 'FORWARD'
            elif steering == 'left':
                robot.left(TURN_SPEED)
                action = 'LEFT'
            elif steering == 'right':
                robot.right(TURN_SPEED)
                action = 'RIGHT'
            else:
                robot.stop()
                action = 'STOP'

        # 3. Annotate and display
        disp = cv2.resize(frame, None, fx=0.3, fy=0.3)
        cv2.putText(disp, f'{action}  {conf:.2f}',
                    (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        display_color.value = bgr8_to_jpeg(disp)
        status_label.value  = f'Frame {frame_idx+1}/{RUN_FRAMES}  Action: {action}  Conf: {conf:.2f}'

except KeyboardInterrupt:
    print('Interrupted by user.')
finally:
    robot.stop()
    print('Robot stopped.')

In [ ]:
# ── Cell 4: Stop camera (always run this when done) ───────────────────────────
robot.stop()
camera.stop()
print('Camera and robot stopped.')